In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.ticker as ticker
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# =============================================================================
# 1. Publication-Quality Visualization Settings (Nature Portfolio Standard)
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# =============================================================================
# 2. Robust Data Loading & Strict Exclusion Rules
# =============================================================================
def clean_and_convert_to_nan(df, cols):
    """
    Strictly converts non-numeric values to np.nan to avoid arbitrary zero-imputation.
    """
    for col in cols:
        df[col] = df[col].astype(str).replace(['Undetermined', '-', 'nan', '#VALUE!', ''], np.nan)
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Load datasets
df_ph = pd.read_csv('pH.csv')
donor_cols = [c for c in df_ph.columns if c.startswith('HS-')]

# Apply strict cleaning
df_ph = clean_and_convert_to_nan(df_ph, donor_cols)

# =============================================================================
# 3. Calculate Delta (Inulin - Control)
# =============================================================================
ctrl_ph = df_ph[df_ph['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0]
inulin_ph = df_ph[df_ph['KULFFI'].str.strip() == 'Inulin'][donor_cols].iloc[0]

# Create DataFrame
df_plot = pd.DataFrame({
    'Donor': donor_cols,
    'Delta_pH': inulin_ph - ctrl_ph
})

# Enforce strict exclusion for missing values
df_plot = df_plot.dropna(subset=['Delta_pH'])

# =============================================================================
# 4. Ecological Stratification (Tertile Thresholds as per 6d)
# =============================================================================
def classify_ecotype(val):
    if val <= -0.80:
        return 'Severe'
    elif -0.80 < val < -0.44:
        return 'Moderate'
    else:
        return 'Mild'

df_plot['Ecotype'] = df_plot['Delta_pH'].apply(classify_ecotype)

# =============================================================================
# 5. Sorting for Waterfall Plot (Ascending order)
# =============================================================================
# Sort donors by Delta_pH in ascending order (Most severe drop on the left, Mild on the right)
df_plot = df_plot.sort_values(by='Delta_pH', ascending=True).reset_index(drop=True)

# Define Color Palette (Matching Fig 6d and 6e)
palette = {'Mild': '#d62728', 'Moderate': '#7f7f7f', 'Severe': '#1f77b4'}
# Map colors to the sorted dataframe
bar_colors = df_plot['Ecotype'].map(palette)

# =============================================================================
# 6. Plotting Figure 6f (pH Stratification Waterfall)
# =============================================================================
fig, ax = plt.subplots(figsize=(8, 5))

# Plot the bars (Y-axis: Delta_pH, Z-order: 2)
bars = ax.bar(df_plot.index, df_plot['Delta_pH'],
              color=bar_colors, width=0.8, edgecolor='black', linewidth=0.5, alpha=0.9, zorder=2)

# Baseline reference (Z-order: 3, to show above bars if needed)
ax.axhline(0, color='black', linestyle='-', linewidth=1.5, zorder=3)

# Add the 33% / 66% (Tertile) ecological threshold lines matching 6d
# Z-order: 1 (Behind the bars) to look clean
ax.axhline(-0.80, color=palette['Severe'], linestyle='--', linewidth=1.5, alpha=0.8, zorder=1)
ax.axhline(-0.44, color=palette['Mild'], linestyle='--', linewidth=1.5, alpha=0.8, zorder=1)

# Axis Labels
ax.set_ylabel(r'$\Delta\,$pH', fontsize=16, fontweight='bold', labelpad=10)
ax.set_xlabel('Individual Donors (n = {})\n(Sorted by '.format(len(df_plot)) + r'$\Delta\,$pH)',
              fontsize=14, fontweight='bold', labelpad=10)

# Aesthetics
# Y-axis ticks every 0.2 increments to match 6d
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.tick_params(axis='y', labelsize=12, width=1.5, length=6)

# Remove X-axis ticks
ax.set_xticks([])

for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Custom Legend
legend_elements = [
    Patch(facecolor=palette['Mild'], edgecolor='black', linewidth=1, label='Mild'),
    Patch(facecolor=palette['Moderate'], edgecolor='black', linewidth=1, label='Moderate'),
    Patch(facecolor=palette['Severe'], edgecolor='black', linewidth=1, label='Severe')
]

# Place legend cleanly outside or at bottom
ax.legend(handles=legend_elements, title='Ecotype (Inulin)',
          title_fontsize=12, fontsize=11, frameon=False,
          loc='lower right', bbox_to_anchor=(1.0, 0.05))

plt.tight_layout()

# Export
output_file = 'Figure_6e.pdf'
plt.savefig(output_file, dpi=DPI_SETTING, bbox_inches='tight')
print(f"Figure successfully saved as {output_file}")